# Claude API Learning Journey

## Step 1 — First API Request
Install dependencies, load API key from `.env`, initialise the Anthropic client, and make the first `client.messages.create()` call.

## Step 2 — Multi-Turn Conversations
Claude is stateless — maintain a `messages` list and send the full history with every request. Helper functions: `add_user_message()`, `add_assistant_message()`, `chat()`.

## Step 3 — Chat Exercise: Continuous Chat Loop
`while True` loop that accepts user input, sends the full conversation history to Claude, appends the response, and repeats — creating a stateful terminal chat session.

## Step 4 — System Prompts
Shape Claude's role and behaviour with a `system` parameter. Updated `chat()` to accept an optional `system` arg (conditionally included — API rejects `system=None`).

## Step 5 — System Prompts Exercise
Applied a one-sentence system prompt `"You are a Python engineer who writes very concise code"` — Claude returned a 1-line solution vs. a verbose multi-function response without the prompt.

## Step 6 — Temperature
`temperature` (0.0–1.0) controls token selection randomness. Low → deterministic/factual. High → creative/varied. Added as an optional param to `chat()` defaulting to `1.0`. Verified with movie idea generation — temperature 0 produced near-identical responses across 3 runs, temperature 1 produced distinct ideas each time.

## Step 7 — Response Streaming
Stream Claude's response word by word using `stream=True` or `client.messages.stream()`. Three patterns: raw events, simplified `text_stream`, and `get_final_message()` for the assembled result after streaming.

## Step 8 — Structured Data
Force clean structured output (JSON, code, CSV) without markdown wrappers or explanatory prose. Course technique: assistant message prefill + stop sequences — not supported on Claude 4. Workaround: system prompt instructing raw output directly.

## Step 9 — Structured Data Exercise
Applied the system prompt workaround to generate raw AWS CLI bash commands. Course prefill approach (` ```bash ` + stop sequence) fails on Claude 4 — replaced with `system="Output raw bash commands only..."` for compatible clean output.

## Step 10 — Prompt Evaluation: Generating Test Datasets
Auto-generate eval datasets using Claude itself. Course technique uses assistant prefill + stop sequences for raw JSON — not supported on Claude 4. Fixed with system prompt workaround. Used `json.dumps(dataset, indent=2)` for readable output. Dataset saved to `dataset.json` for reuse across eval runs.

## Step 11 — Prompt Evaluation: Running the Eval
Built the core eval pipeline — three functions: `run_prompt()` merges a test case into the prompt and calls Claude; `run_test_case()` runs the prompt and returns `{output, test_case, score}`; `run_eval()` loops over the full dataset and collects all results. Grading is hardcoded to 10 as a placeholder — real grading logic added in the next step.

## Step 12 — Prompt Evaluation: Model Based Grading
Replaced the hardcoded score with a real model grader. `grade_by_model()` sends the task and Claude's output to a second Claude call that returns a JSON object with `strengths`, `weaknesses`, `reasoning`, and `score`. Applied the Claude 4 system prompt fix (no assistant prefill). `run_eval()` now prints an average score across all test cases.

In [114]:
# Install Dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [115]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [116]:
# Create an API client
from anthropic import Anthropic 
client = Anthropic()
model = "claude-sonnet-4-6"

In [117]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
        
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    return message.content[0].text

In [118]:
import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

It should only have "task" property in the output json array items.

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects. Return a raw JSON array only — no markdown, no explanation, no code fences.
"""
    messages = []
    add_user_message(messages, prompt)
    # Claude 4 doesn't support assistant prefill — enforce raw JSON via system prompt instead
    text = chat(
        messages,
        system="Output raw JSON only. No markdown, no explanation, no code fences."
    )
    return json.loads(text)

dataset = generate_dataset()
print(json.dumps(dataset, indent=2))

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

[
  {
    "task": "Write a Python function that takes an AWS S3 bucket name and a prefix string, and returns a list of all object keys in that bucket matching the given prefix using the boto3 client."
  },
  {
    "task": "Write a JSON object representing an AWS IAM policy that allows a user to list and get objects from a specific S3 bucket named 'my-data-bucket' but denies the ability to delete any objects."
  },
  {
    "task": "Write a regular expression that matches a valid AWS ARN (Amazon Resource Name) in the format arn:partition:service:region:account-id:resource, where each segment is non-empty and separated by colons."
  }
]


In [119]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [120]:
def grade_by_model(task_case, output):
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task_case["task"]}
    
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(
        messages,
        system="Output raw JSON only. No markdown, no explanation, no code fences."
    )
    return json.loads(eval_text)

In [121]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }

In [122]:
from statistics import mean
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
     
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    
    return results

In [123]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(results)

print(json.dumps(results, indent=2))

Average score: 7
[{'output': '## AWS S3 Object Key Listing Function\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError, NoCredentialsError\n\n\ndef list_s3_objects(bucket_name: str, prefix: str = "") -> list[str]:\n    """\n    List all object keys in an S3 bucket matching the given prefix.\n\n    Args:\n        bucket_name: The name of the S3 bucket.\n        prefix: The prefix string to filter objects (default: "" lists all objects).\n\n    Returns:\n        A list of object keys matching the given prefix.\n\n    Raises:\n        ValueError: If bucket_name is empty.\n        NoCredentialsError: If AWS credentials are not configured.\n        ClientError: If there\'s an AWS API error (e.g., bucket not found, access denied).\n    """\n    if not bucket_name:\n        raise ValueError("bucket_name cannot be empty.")\n\n    s3_client = boto3.client("s3")\n    object_keys = []\n\n    try:\n        paginator = s3_client.get_paginator("list_objects_v2")\n        page_it